# WaterOnt: LLM comparison with dense retrieval

Runs the three LLM systems together, saving every answer: flat-text RAG with TF-IDF retrieval, flat-text RAG with **dense-vector retrieval** (`BAAI/bge-small-en-v1.5`), and WaterOnt GraphRAG. It also measures what TF-IDF, BM25 and dense retrieval put in the LLM's context. Results go to new `retrieval_*` files; the earlier result files are not touched.

Besides the Table 5 metrics, the run reports **policy precision**: the share of returned (pipe, action) pairs that are correct under the policy, without the top-25 cap of the reference answers.

**Before you start**
1. Runtime → **Disconnect and delete runtime**.
2. Your `ANTHROPIC_API_KEY` secret is already set (notebook access on).
3. Have the new `WaterOnt-full-repo.zip` on your computer; step 2 asks for it.
4. Run the cells in order. Step 1 may restart the runtime once; if so, continue from step 2. Continue past step 3 only if it prints `API replied: OK`.

In [ ]:
# 1. Install packages (keeps NumPy compatible with Colab's SciPy)
import os, subprocess, sys
os.chdir("/content")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=2.0,<2.3",
                "wntr", "rdflib", "anthropic", "sentence-transformers"], check=True)

on_disk = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
import numpy
if numpy.__version__ != on_disk:
    print(f"NumPy changed ({numpy.__version__} -> {on_disk}): restarting. Continue from step 2.")
    os.kill(os.getpid(), 9)
print("Step 1 done. NumPy", numpy.__version__)

In [ ]:
# 2. Get the code from WaterOnt-full-repo.zip
import os, sys, shutil, zipfile
from google.colab import files
os.chdir("/content")
uploaded = files.upload()                       # choose WaterOnt-full-repo.zip
shutil.rmtree("/content/WaterOnt", ignore_errors=True)
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall("/content")
os.chdir("/content/WaterOnt")
if "/content/WaterOnt" not in sys.path:
    sys.path.insert(0, "/content/WaterOnt")
for name in [m for m in sys.modules if m.startswith("wateront")]:
    del sys.modules[name]

import inspect
from wateront.retrieval import DENSE_MODEL
from wateront.evaluation import reference
from wateront.config import Config
script = open("paper/experiments_retrieval.py").read()
if "capped" not in inspect.signature(reference).parameters or "KG, LLM (WaterOnt GraphRAG)" not in script:
    raise RuntimeError("This is an OLD WaterOnt-full-repo.zip. Download the latest one and rerun step 2.")
print("Step 2 done. Latest code: runs TF-IDF RAG, dense RAG and GraphRAG, saves answers, reports policy precision.")
print("LLM model:", Config().llm_model, "| dense model:", DENSE_MODEL)

In [ ]:
# 3. API key and a one-line test call. Must print "API replied: OK".
import os, anthropic
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip().strip("\"'")
reply = anthropic.Anthropic().messages.create(
    model=Config().llm_model, max_tokens=10,
    messages=[{"role": "user", "content": "Reply with OK"}])
print("Step 3 done. API replied:", reply.content[0].text)

In [ ]:
# 4. Smoke test: dense retrieval on a question that TF-IDF could not serve
from wateront.pipeline import build_knowledge_graph
from wateront.baselines import FlatRAGBaseline
from wateront.retrieval import sentence_transformer_embedder, DENSE_QUERY_PREFIX

cfg = Config(seed=42)
res = build_knowledge_graph("data/networks/L-TOWN.inp", cfg, verbose=False)
embed = sentence_transformer_embedder(DENSE_MODEL)          # downloads the model once
question = "Which pipes should be replaced, and which standard authorises each replacement?"

dense = FlatRAGBaseline(res.kg, cfg, use_llm=True, retriever="dense", embed=embed, query_prefix=DENSE_QUERY_PREFIX)
records = [d["id"] for d, _ in dense.index.search(question, k=40) if d["kind"] == "asset"]
print("Pipe records supplied by dense retrieval:", len(records), "(TF-IDF supplied none for this question)")
answer = dense.ask(question)
print("Recommendations:", len(answer.recommendations), "| unsupported citations:", answer.unsupported_citations)
print(answer.text[:400])

Step 4 should report about 30 to 40 pipe records. Then run the full experiment (it reruns WaterOnt GraphRAG too, so all LLM results come from one run).

In [ ]:
# 5. Full run: retrieval analysis, then the three LLM systems (about 20-30 minutes, about 100 API calls)
import subprocess, sys
run = subprocess.run([sys.executable, "paper/experiments_retrieval.py"], capture_output=True, text=True,
                     cwd="/content/WaterOnt")
print(run.stdout[-3000:])
if run.returncode != 0:
    print(run.stderr[-3000:])
    raise RuntimeError("experiments_retrieval.py failed; send the error above")
print("Step 5 done.")

In [ ]:
# 6. Results
import json, pandas as pd
meta = json.load(open("paper/results/retrieval_meta.json"))
print("LLM run:", meta["llm_run"], "| planning failures:", meta.get("llm_planning_failures"))
print("Questions without a recommendation:", meta.get("no_recommendation_questions"))
ra = pd.read_csv("paper/results/retrieval_analysis.csv")
print("\nRetrieval (no LLM):")
display(ra.groupby("retriever", sort=False).agg(
    no_pipe_records=("pipe_records", lambda x: int((x == 0).sum())),
    records_min=("pipe_records", "min"), records_max=("pipe_records", "max"),
    reference_in_context=("reference_in_context", "sum"), reference_pipes=("reference_pipes", "sum")))
print("\nLLM systems in this run (Table 5 metrics plus policy precision):")
display(pd.read_csv("paper/results/retrieval_rag_summary.csv").set_index("system").round(3))

In [ ]:
# 7. Download the new result files
import shutil
from google.colab import files
os.makedirs("/content/dense_results", exist_ok=True)
for f in ["retrieval_analysis.csv", "retrieval_rag_per_query.csv", "retrieval_rag_summary.csv", "retrieval_meta.json"]:
    shutil.copy(f"/content/WaterOnt/paper/results/{f}", "/content/dense_results/")
shutil.make_archive("/content/dense_results", "zip", "/content/dense_results")
files.download("/content/dense_results.zip")